# Nobitex API preflight — function-by-function

This notebook tests the **actual functions in the `.py` files** before Real Mode.

Safety order:

1. Offline authentication helpers
2. Offline pytest regression tests
3. Public market data
4. Authenticated **read-only** account calls
5. Portfolio calls
6. Real order/cancel functions with HTTP **mocked**
7. Demo/paper order lifecycle
8. Optional **Testnet** add/cancel (disabled by default)

**No production/mainnet order is intentionally submitted by this notebook.**
Do not paste API credentials into a cell; use the `getpass()` cell below.

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
loaded = load_dotenv(".env", override=True)

print("dotenv loaded:", loaded)
print("API key exists:", bool(os.environ.get("NOBITEX_API_KEY")))
print("Secret key exists:", bool(os.environ.get("NOBITEX_SECRET_KEY")))

## 0. Project path and Python environment

In [ ]:
from pathlib import Path
import os, sys

HERE = Path.cwd().resolve()
if (HERE / "exchange").is_dir() and (HERE / "config.py").exists():
    MM_PATH = HERE
elif (HERE / "market_maker").is_dir():
    MM_PATH = HERE / "market_maker"
else:
    # Server path used in the current project. Change only if you moved the folder.
    MM_PATH = Path("/root/jupyter-workspace/MM/nobitex_project/lob_modified/market_maker")

assert MM_PATH.is_dir(), f"market_maker folder not found: {MM_PATH}"
os.chdir(MM_PATH)
if str(MM_PATH) not in sys.path:
    sys.path.insert(0, str(MM_PATH))

print("Python:", sys.executable)
print("market_maker:", MM_PATH)
print("cwd:", Path.cwd())

In [ ]:
import importlib.util

required = [
    "requests", "cryptography", "pandas", "numpy", "pytz", "dotenv",
    "pytest", "mysql.connector"
]

def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ModuleNotFoundError, ImportError):
        return False

missing = [m for m in required if not has_module(m)]
print("Missing:", missing if missing else "none")
if missing:
    print("Run once in a new cell:  %pip install -r ../requirements-dev.txt")

## 1. Import the real project modules

In [ ]:
from exchange.nobitex_auth import (
    build_full_path, compact_json, create_signature, signed_headers
)
from exchange.nobitex import NobitexExchangeClient
from order_status import OrderStatusChecker, OrderCanceller
from portfolio import Nobitex_portfolio
from exchange.demo import DemoExchangeClient
from paper import PaperLedger

import exchange.nobitex_auth as auth_module
import exchange.nobitex as exchange_module
import order_status as order_status_module
import portfolio as portfolio_module

print("auth:", auth_module.__file__)
print("exchange:", exchange_module.__file__)
print("order_status:", order_status_module.__file__)
print("portfolio:", portfolio_module.__file__)

## 2. Offline authentication test — no network

In [ ]:
import base64
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.hazmat.primitives.serialization import Encoding, PrivateFormat, NoEncryption

private = Ed25519PrivateKey.generate()
raw_secret = private.private_bytes(Encoding.Raw, PrivateFormat.Raw, NoEncryption())
fake_secret = base64.urlsafe_b64encode(raw_secret).decode("ascii").rstrip("=")

body = compact_json({"currency": "rls"})
assert body == '{"currency":"rls"}'

full_path = build_full_path(
    "/market/orders/list",
    {"status": "open", "details": 2, "page": 1, "pageSize": 100},
)
assert "?" in full_path and "status=open" in full_path

ts, sig = create_signature(fake_secret, "POST", "/users/wallets/balance", body)
headers = signed_headers("fake-public-key", fake_secret, "POST", "/users/wallets/balance", body)
assert headers["Nobitex-Timestamp"] == ts or headers["Nobitex-Timestamp"].isdigit()
assert headers["Nobitex-Signature"]

print("PASS — compact JSON, query-path construction and Ed25519 signing work")
print("Example signed GET path:", full_path)

## 3. Run the API-auth regression tests — no live orders

In [ ]:
import subprocess

cmd = [
    sys.executable, "-m", "pytest", "-q",
    "tests/test_nobitex_api_key_auth.py",
    "tests/test_real_exchange_defenses.py",
    "tests/test_config_env.py",
    "tests/test_demo_safety.py",
    "tests/test_demo_entrypoint.py",
    "tests/test_demo_config.py",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, text=True)
assert result.returncode == 0, "Targeted preflight pytest suite failed"
print("PASS — targeted regression suite")

## 4. Public live market-data test — no credentials, no orders

In [ ]:
public_client = NobitexExchangeClient(
    api_key=None,
    secret_key=None,
    url_send_order=None,
    url_wallet_balance=None,
    trading_enabled=False,
)

PUBLIC_SYMBOL = "BTCIRT"
orderbook = public_client.get_orderbook(PUBLIC_SYMBOL)
assert orderbook is not None, "Could not read public orderbook"
assert orderbook.get("bids") and orderbook.get("asks")
print("PASS — public orderbook")
print("lastTradePrice:", orderbook.get("lastTradePrice"))
print("best bid:", orderbook["bids"][0])
print("best ask:", orderbook["asks"][0])

## 5. Load production credentials in memory — read-only testing

The next cells call authenticated endpoints but do not place/cancel orders.
Use a key with at least `READ` permission. `getpass()` prevents the values from being printed or saved as cell source.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(".env", override=True)

API_KEY = os.environ.get("NOBITEX_API_KEY")
SECRET_KEY = os.environ.get("NOBITEX_SECRET_KEY")

assert API_KEY
assert SECRET_KEY

print("API key loaded:", bool(API_KEY))
print("Secret key loaded:", bool(SECRET_KEY))

In [ ]:
PROD_BASE_URL = "https://apiv2.nobitex.ir"
readonly_client = NobitexExchangeClient(
    api_key=API_KEY,
    secret_key=SECRET_KEY,
    url_send_order=f"{PROD_BASE_URL}/market/orders/add",
    url_wallet_balance=f"{PROD_BASE_URL}/users/wallets/balance",
    trading_enabled=False,  # hard-blocks send/cancel methods
    base_url=PROD_BASE_URL,
)
assert readonly_client.trading_enabled is False
print("PASS — authenticated client created with write methods blocked")

### 5.1 Wallet balance — live, read-only

In [ ]:
READ_CURRENCIES = ["rls", "btc", "usdt"]
wallet_results = {}
for currency in READ_CURRENCIES:
    value = readonly_client.wallet_balance(currency)
    wallet_results[currency] = value
    print(currency.upper(), "=>", value)
    assert value is not None, f"wallet_balance failed for {currency}"
print("PASS — wallet_balance()")

### 5.2 User order list — live, read-only

In [ ]:
open_orders = readonly_client.get_order_list_status(
    status="open", details=2, page_size=100, max_pages=1
)
assert isinstance(open_orders, list)
print("PASS — get_order_list_status(open)")
print("open order count on first page:", len(open_orders))
if open_orders:
    safe_preview = {
        k: open_orders[0].get(k)
        for k in ["id", "status", "type", "clientOrderId", "pair", "price", "amount"]
    }
    print("first order preview:", safe_preview)

### 5.3 Trade timestamp lookup — live, read-only, optional

In [ ]:
# Set this to a real order ID only if you want to test get_order_done_time().
# Leaving it as None safely skips the call.
ORDER_ID_FOR_TRADE_TEST = None

if ORDER_ID_FOR_TRADE_TEST is None:
    print("SKIP — set ORDER_ID_FOR_TRADE_TEST to test trade-history lookup")
else:
    stamp = readonly_client._status_checker.get_order_done_time(ORDER_ID_FOR_TRADE_TEST)
    print("trade timestamp:", stamp)
    print("PASS — get_order_done_time() request completed")

## 6. Portfolio functions — live, read-only

In [ ]:
portfolio = Nobitex_portfolio(API_KEY, SECRET_KEY, base_url=PROD_BASE_URL)
assert "Authorization" not in portfolio.session.headers
assert "Nobitex-Key" not in portfolio.session.headers
print("PASS — portfolio Session has no stale auth headers")

In [ ]:
for currency in ["btc", "usdt", "rls"]:
    bal = portfolio.get_balance(currency)
    print(f"get_balance({currency!r}) =>", bal)
print("PASS — get_balance()")

In [ ]:
balances = portfolio.get_balances(["btc", "usdt", "rls"])
print("get_balances =>", balances)
assert set(balances) == {"BTC", "USDT", "RLS"}
print("PASS — get_balances()")

In [ ]:
price = portfolio.get_best_price("USDTIRT")
print("USDTIRT best-effort price =>", price)
assert price is not None and price > 0
print("PASS — get_best_price()")

In [ ]:
snapshot = portfolio.portfolio_snapshot_multi(
    ["btc", "usdt"], quote="IRT", include_cash=True
)
print("quote:", snapshot["quote"])
print("cash wallet:", snapshot["cash_wallet_ccy"])
print("assets:", list(snapshot["assets"]))
print("total_value_irt:", snapshot["total_value_irt"])
assert "btc" in snapshot["assets"] and "usdt" in snapshot["assets"]
print("PASS — portfolio_snapshot_multi()")

## 7. `send_order()` write path — HTTP mocked

This calls the **real `NobitexExchangeClient.send_order()` function**, but `requests.post` is intercepted. No HTTP request leaves Python and no order can reach Nobitex.

In [ ]:
import time

from config import MMConfig, TRADING_PROFILES
from exchange.nobitex import NobitexExchangeClient


# ============================================================
# 1. SELECT CONFIG PROFILE
# ============================================================

PROFILE = "usdt"

assert PROFILE in TRADING_PROFILES

cfg = MMConfig(
    **TRADING_PROFILES[PROFILE]
)


# ============================================================
# 2. CHECK CONFIG
# ============================================================

assert cfg.api_key, "API key is missing from config/.env"
assert cfg.secret_key, "Secret key is missing from config/.env"

assert cfg.url_send_order, "url_send_order is missing"
assert cfg.url_wallet_balance, "url_wallet_balance is missing"

print("Profile:", PROFILE)
print("Pair:", cfg.pair)
print("Source currency:", cfg.currency_1)
print("Destination currency:", cfg.currency_2)
print("Configured order size:", cfg.order_size)
print("Algorithm ID:", cfg.algorithm_id)
print("Order endpoint:", cfg.url_send_order)

print()
print("API key loaded:", bool(cfg.api_key))
print("Secret key loaded:", bool(cfg.secret_key))




In [ ]:
import json
from unittest.mock import patch


class MockResponse:
    def __init__(self, body, status_code=200):
        self.status_code = status_code
        self._body = body

    def json(self):
        return self._body

    def raise_for_status(self):
        if not 200 <= self.status_code < 300:
            raise RuntimeError(f"HTTP {self.status_code}")


write_client = NobitexExchangeClient(
    api_key="fake-public-key",
    secret_key=fake_secret,
    url_send_order="https://mock.invalid/market/orders/add",
    url_wallet_balance="https://mock.invalid/users/wallets/balance",
    algorithm_id="preflight",
    trading_enabled=True,
)
captured_send = {}


def fake_send_post(url, *, headers, data, timeout):
    captured_send["url"] = url
    captured_send["headers"] = headers
    captured_send["body"] = data
    return MockResponse({"status": "ok", "order": {"id": 123, "clientOrderId": "preflight-order-1"}})


# send_order() calls requests.post inside exchange.nobitex; intercept it there.
with patch.object(exchange_module.requests, "post", side_effect=fake_send_post):
    returned_id = write_client.send_order(
        "buy", "btc", "rls", 0.001, 100_000, clientOrderId="preflight-order-1"
    )

assert returned_id == "preflight-order-1", returned_id
assert captured_send["url"] == "https://mock.invalid/market/orders/add"
assert json.loads(captured_send["body"]) == {
    "type": "buy", "srcCurrency": "btc", "dstCurrency": "rls",
    "amount": "0.001", "price": 100_000, "clientOrderId": "preflight-order-1",
}
assert captured_send["headers"]["Nobitex-Key"] == "fake-public-key"
assert captured_send["headers"]["Nobitex-Signature"]
assert "Authorization" not in captured_send["headers"]

# A rejected or mismatched reply must come back as None, never as success.
with patch.object(exchange_module.requests, "post",
                  return_value=MockResponse({"status": "failed"}, status_code=400)):
    assert write_client.send_order(
        "buy", "btc", "rls", 0.001, 100_000, clientOrderId="preflight-order-2"
    ) is None
with patch.object(exchange_module.requests, "post",
                  return_value=MockResponse({"status": "ok", "order": {"clientOrderId": "someone-else"}})):
    assert write_client.send_order(
        "buy", "btc", "rls", 0.001, 100_000, clientOrderId="preflight-order-3"
    ) is None

# The read-only client from section 5 must refuse to send at all.
try:
    readonly_client.send_order("buy", "btc", "rls", 0.001, 100_000, clientOrderId="blocked")
except Exception as exc:
    print("read-only client refused to send:", type(exc).__name__)
else:
    raise AssertionError("read-only client sent an order")

print("PASS — real send_order() logic with API-key auth; network mocked")
print("raw body sent:", captured_send["body"])


## 8. `cancel_order()` write path — HTTP mocked

In [ ]:
canceller = OrderCanceller("fake-key", fake_secret, base_url="https://mock.invalid")
captured_cancel = {}

def fake_cancel_post(url, *, headers, data, timeout):
    captured_cancel["url"] = url
    captured_cancel["headers"] = headers
    captured_cancel["body"] = data
    return MockResponse({"status": "ok", "order": {"id": 123, "clientOrderId": "preflight-order-1"}})

with patch.object(canceller.session, "post", side_effect=fake_cancel_post):
    cancel_response = canceller.cancel_order(123)

assert cancel_response["status"] == "ok"
assert json.loads(captured_cancel["body"]) == {"status": "canceled", "order": 123}
assert "Authorization" not in canceller.session.headers
assert "Nobitex-Key" in captured_cancel["headers"]
print("PASS — real cancel_order() logic with API-key auth; network mocked")
print("raw body sent:", captured_cancel["body"])

In [ ]:
open_orders = readonly_client.get_order_list_status(
    status="open", details=2, page_size=100, max_pages=1
)
assert isinstance(open_orders, list)
print("PASS — get_order_list_status(open)")
print("open order count on first page:", len(open_orders))
if open_orders:
    safe_preview = {
        k: open_orders[0].get(k)
        for k in ["id", "status", "type", "clientOrderId", "pair", "price", "amount"]
    }
    print("first order preview:", safe_preview)

## 9. Scoped bulk-cancel and stale-cancel logic — mocked dependencies

In [ ]:
from datetime import datetime, timedelta, timezone

class FakeChecker:
    def __init__(self, orders):
        self.orders = list(orders)
    def get_order_list_status(self, **kwargs):
        return list(self.orders)

class FakeCanceller:
    def __init__(self):
        self.cancelled = []
    def cancel_order(self, order_id):
        self.cancelled.append(order_id)
        return {"status": "ok", "order": {"id": order_id}}

now = datetime.now(timezone.utc)
old = (now - timedelta(hours=2)).isoformat()
fresh = now.isoformat()
orders = [
    {"id": 1, "clientOrderId": "owned-old", "pair": "BTCIRT", "created_at": old},
    {"id": 2, "clientOrderId": "owned-fresh", "pair": "BTCIRT", "created_at": fresh},
    {"id": 3, "clientOrderId": "foreign-old", "pair": "BTCIRT", "created_at": old},
    {"id": 4, "clientOrderId": "other-pair", "pair": "USDTIRT", "created_at": old},
]
logic_client = NobitexExchangeClient(
    "fake-key", fake_secret,
    "https://mock.invalid/market/orders/add",
    "https://mock.invalid/users/wallets/balance",
    trading_enabled=True,
)
logic_client._status_checker = FakeChecker(orders)
logic_client._canceller = FakeCanceller()

cancelled = logic_client.cancel_all_open_orders(
    pair="BTCIRT", owned_client_order_ids={"owned-old", "owned-fresh"}
)
assert cancelled == {"owned-old", "owned-fresh"}
assert logic_client._canceller.cancelled == [1, 2]
print("PASS — cancel_all_open_orders() remains pair + ownership scoped")

logic_client._canceller = FakeCanceller()
tracked = {"buy": {"owned-old", "owned-fresh"}, "sell": set()}
cancelled_stale = logic_client.cancel_old_orders(
    30,
    pair="BTCIRT",
    owned_client_order_ids={"owned-old", "owned-fresh"},
    tracked_orders=tracked,
    now=now,
)
assert cancelled_stale == {"owned-old"}
assert logic_client._canceller.cancelled == [1]
assert tracked["buy"] == {"owned-fresh"}
print("PASS — cancel_old_orders() cancels only owned, stale orders for the selected pair")

## 10. Demo/paper order lifecycle — no real order endpoint

In [ ]:
from datetime import datetime, timezone

ledger = PaperLedger(
    session_id="preflight",
    symbol="BTCIRT",
    initial_inventory=1.0,
    initial_cash=1_000_000_000.0,
    commission_perc=0.001,
    valuation_price=100_000.0,
)
demo = DemoExchangeClient(
    market_data=public_client,
    pair="BTCIRT",
    commission_perc=0.001,
    ledger=ledger,
)

cid = demo.send_order(
    "buy", "btc", "rls", 0.001, 100_000,
    clientOrderId="paper-preflight-1",
    selected_level=0,
)
assert cid == "paper-preflight-1"
open_demo = demo.get_order_list_status("open")
assert any(o["clientOrderId"] == cid for o in open_demo)
print("PASS — DemoExchangeClient.send_order() created a paper order")

resp = demo.cancel_order(cid)
assert resp["status"] == "ok"
assert not demo.get_order_list_status("open")
print("PASS — DemoExchangeClient.cancel_order() cancelled the paper order")

## 11. Build the real application in Demo Mode

In [ ]:
from run import make_cfg, make_bot

cfg = make_cfg("btc", mode="demo", session_id="preflight-notebook")
bot = make_bot(cfg, algorithm="algo1", exchange_name="nobitex", mode="demo")
assert bot.exchange.is_demo is True
assert bot.exchange._market_data.trading_enabled is False
assert bot.exchange._market_data.api_key is None
assert bot.exchange._market_data.secret_key is None
print("PASS — run.py make_cfg/make_bot Demo path strips authenticated trading capability")

## 12. Optional Testnet end-to-end add/cancel

Nobitex documents a Testnet API at `https://testnetapi.nobitex.ir`.
This section is **disabled by default**. It will place and then attempt to cancel a Testnet order, never a production order.

Create/use **Testnet credentials**, set the order parameters yourself, and only then set `RUN_TESTNET_WRITE_TESTS = True`.

In [ ]:
RUN_TESTNET_WRITE_TESTS = False

TESTNET_SRC = None       # e.g. "btc"
TESTNET_DST = None       # e.g. "rls"
TESTNET_AMOUNT = None    # choose a valid Testnet amount
TESTNET_PRICE = None     # choose a valid Testnet limit price

if not RUN_TESTNET_WRITE_TESTS:
    print("SKIP — Testnet write test is disabled")
else:
    assert all(v is not None for v in [TESTNET_SRC, TESTNET_DST, TESTNET_AMOUNT, TESTNET_PRICE])
    TESTNET_API_KEY = getpass("Testnet Nobitex API Key: ").strip()
    TESTNET_SECRET_KEY = getpass("Testnet Nobitex Secret Key: ").strip()
    TESTNET_BASE = "https://testnetapi.nobitex.ir"

    testnet_client = NobitexExchangeClient(
        api_key=TESTNET_API_KEY,
        secret_key=TESTNET_SECRET_KEY,
        url_send_order=f"{TESTNET_BASE}/market/orders/add",
        url_wallet_balance=f"{TESTNET_BASE}/users/wallets/balance",
        algorithm_id="preflight-testnet",
        trading_enabled=True,
        base_url=TESTNET_BASE,
    )
    test_client_id = f"preflight-testnet-{int(__import__('time').time()*1000)}"
    result_id = testnet_client.send_order(
        "buy", TESTNET_SRC, TESTNET_DST,
        TESTNET_AMOUNT, TESTNET_PRICE,
        clientOrderId=test_client_id,
    )
    assert result_id == test_client_id, "Testnet order was not confirmed"
    print("Testnet order submitted:", result_id)

    orders = testnet_client.get_order_list_status("open", max_pages=2)
    match = next((o for o in orders if str(o.get("clientOrderId")) == result_id), None)
    assert match is not None, "Submitted Testnet order not found in open-order list"
    assert match.get("id") is not None

    cancel = testnet_client.cancel_order(match["id"])
    assert cancel.get("status") in {"ok", "success"}, cancel
    print("PASS — Testnet add -> list -> cancel end-to-end")

## 12b. Optional production add/cancel — disabled by default

This is the only cell that can place a **real** order on `apiv2.nobitex.ir`. It sends one limit order with the chosen profile's `order_size`, priced away from the best price so it rests, confirms it is in the open-order list, and cancels it again.

It does nothing unless you set `RUN_REAL_ORDER_TEST = True` **and** type `SEND` at the prompt. Headless runs (`jupyter nbconvert --execute`) cannot answer the prompt, so they can never send it.

In [ ]:
RUN_REAL_ORDER_TEST = False

REAL_PROFILE = "usdt"       # trading profile whose pair / order_size are used
REAL_ORDER_TYPE = "buy"
REAL_PRICE_OFFSET = -500    # rial added to the best price (negative = below it)

if not RUN_REAL_ORDER_TEST:
    print("SKIP — production add/cancel test is disabled")
else:
    import time
    from config import MMConfig, TRADING_PROFILES

    real_cfg = MMConfig(**TRADING_PROFILES[REAL_PROFILE])
    assert real_cfg.api_key and real_cfg.secret_key, "API keys missing from .env"
    real_client = NobitexExchangeClient(
        api_key=real_cfg.api_key,
        secret_key=real_cfg.secret_key,
        url_send_order=real_cfg.url_send_order,
        url_wallet_balance=real_cfg.url_wallet_balance,
        algorithm_id=real_cfg.algorithm_id,
        trading_enabled=True,
        base_url=PROD_BASE_URL,
    )
    amount = real_cfg.order_size
    best = portfolio.get_best_price(real_cfg.pair)
    assert best is not None and best > 0, "no best price available"
    price = float(best) + REAL_PRICE_OFFSET
    side = "b" if REAL_ORDER_TYPE == "buy" else "s"
    client_order_id = f"mm{real_cfg.algorithm_id}{side}{int(time.time() * 1000)}"
    assert len(client_order_id) <= 32, client_order_id

    print("========== REAL ORDER ==========")
    print("pair:", real_cfg.pair, "| type:", REAL_ORDER_TYPE,
          "| amount:", amount, "| price:", price)
    print("clientOrderId:", client_order_id)
    confirmation = input("\nThis will send a REAL order to Nobitex. Type SEND to continue: ")
    assert confirmation == "SEND", "Order test cancelled."

    returned = real_client.send_order(
        order_type=REAL_ORDER_TYPE,
        srcCurrency=real_cfg.currency_1,
        dstCurrency=real_cfg.currency_2,
        amount=amount,
        price=price,
        clientOrderId=client_order_id,
    )
    assert returned == client_order_id, f"send_order() returned {returned!r}"
    print("PASS — Nobitex accepted the order:", returned)

    resting = [o for o in real_client.get_order_list_status("open", max_pages=2)
               if str(o.get("clientOrderId")) == client_order_id]
    assert resting, "the new order is not in the open-order list; cancel it by hand"
    cancel_response = real_client.cancel_order(resting[0]["id"])
    assert cancel_response.get("status") == "ok", cancel_response
    print("PASS — order cancelled again, id:", resting[0]["id"])


## 13. Final gate

Before `python run.py --profile ... --mode real`, all of these should be green:

- Offline signature test: PASS
- Targeted pytest suite: PASS
- Public orderbook: PASS
- `wallet_balance`: PASS
- `get_order_list_status`: PASS
- Portfolio balance/price/snapshot: PASS
- Mocked `send_order`: PASS
- Mocked `cancel_order`: PASS
- Scoped bulk/stale cancellation logic: PASS
- Demo/paper order lifecycle: PASS
- `make_cfg` / `make_bot` Demo safety: PASS
- Optional but recommended Testnet add/list/cancel: PASS

Real Mode is intentionally not launched from this notebook.